In [1]:
import json
import random
import re
import random
from collections import defaultdict
from pathlib import Path
from transformers import AutoTokenizer

ABS_REL_DIR = Path("/home/lucas/Desktop/UCSD/Research/sequential-decision-processors/verl_dead_agent/agent_system/environments")
tok_path = ABS_REL_DIR / "tokenizers" / "qwen3"
tok = AutoTokenizer.from_pretrained(tok_path, use_fast=True, local_files_only=True)

/home/lucas/Desktop/UCSD/Research/sequential-decision-processors/sequential_decision_processors/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## Viewing Our Data

In [2]:
def load_jsonl(path: Path):
    data = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data.append(json.loads(line))
    return data

def count_tokens(tokenizer, inp) -> int:
    if isinstance(inp, str):
        return len(tokenizer.encode(inp))
    return sum(len(tokenizer.encode(x)) for x in inp)

def print_sample(sample: dict):
    for k, v in sample.items():
        print(f"{k}:")
        if k == "info":
            for k_infos, v_infos in sample[k].items():
                if k_infos == "obs":
                    continue
                else:
                    print(f"[{k_infos}]: {v_infos}")
        else:
            print(v)
            print("-" * 80)

def print_sample_token_counts(sample):
    print(f"Token Counts:\n{'-'*60}")
    print(f"Input: {count_tokens(tok, sample['input'])}")
    print(f"Output: {count_tokens(tok, sample['output'])}")
    print(f"{'-'*60}\n")

In [3]:
jsonl_path = Path("rejection_sampling/alfworld_7/1.jsonl")

data = load_jsonl(jsonl_path)
print(f"Loaded {len(data)} samples")

Loaded 541 samples


In [4]:
random_sample = random.choice(data)
print_sample_token_counts(random_sample)
print_sample(random_sample)

Token Counts:
------------------------------------------------------------
Input: 633
Output: 1024
------------------------------------------------------------

input:
user
You are an agent operating in AlfWorld, an interactive-fiction, text-world environment.
You should first reason about your current situation prior to returning your chosen action. You MUST format your thinking as <think> your_reasoning </think> and your action as <action> your_action </action>. 
If you do not enclose your reasoning and action within their respective tags, your response will be rejected. You can only provide one action at a time.
For example, <think> my_thinking... </think> <action> take lantern </action>.
Note that you must move to an object before you can interact with it. You can use 'inventory' to get your current inventory.
The set of action templates are the following: ['go to _', 'open _', 'close _', 'take _ from _', 'move _ to _', 'use _', 'heat _ with _', 'cool _ with _', 'clean _ with _', '

In [5]:
# First group our data
groups = defaultdict(list)
for ex in data:
    run_info = ex.get("run_info", {})
    seed = run_info.get("seed")
    proc_id = run_info.get("proc_id")
    if seed is None or proc_id is None:
        continue
    groups[(seed, proc_id)].append(ex)

# Only keep groups with at least 2 entries (i.e., real “matches”)
matches = {k: v for k, v in groups.items() if len(v) > 1}

# Random sample of match groups to inspect
num_samples = 1
sample_keys = random.sample(list(matches.keys()), min(num_samples, len(matches)))

for (seed, proc_id) in sample_keys:
    traj = sorted(matches[(seed, proc_id)],
                  key=lambda ex: ex["run_info"]["step"])  # sort by env step

    print(f"=== seed={seed}, proc_id={proc_id} ===")
    for ex in traj:
        step = ex["run_info"]["step"]
        print(f"step {step} - score {ex.get('score')}:")
        print(f"Input\n{ex.get('input')}")
        print(f"Output\n{ex.get('output')}")
        print(f"{'-'*60}")

=== seed=/root/.cache/tales/alfworld/json_2.1.1/train/pick_cool_then_place_in_recep-Bowl-None-Cabinet-19/trial_T20190908_022130_590876/game.tw-pddl, proc_id=209350 ===
step 1 - score 0.0:
Input
user
You are an agent operating in AlfWorld, an interactive-fiction, text-world environment.
You should first reason about your current situation prior to returning your chosen action. You MUST format your thinking as <think> your_reasoning </think> and your action as <action> your_action </action>. 
If you do not enclose your reasoning and action within their respective tags, your response will be rejected. You can only provide one action at a time.
For example, <think> my_thinking... </think> <action> take lantern </action>.
Note that you must move to an object before you can interact with it. You can use 'inventory' to get your current inventory.
The set of action templates are the following: ['go to _', 'open _', 'close _', 'take _ from _', 'move _ to _', 'use _', 'heat _ with _', 'cool _ wi